<a href="https://colab.research.google.com/github/Awais-Ahmad10/flyrank_ml_internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Awais-Ahmad10/flyrank_ml_internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a scoring and ranking task. My lane (Refresh / Content Opportunity Scoring) isn't about sorting pages into fixed categories; it's about assigning a continuous score to every page and ordering the entire inventory. This creates a prioritized queue so editorial teams know which decaying or high-potential pages to review first. The output that matters is the ranked list to support content operations, not a simple yes/no label.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target variable is a content opportunity score (a continuous value representing refresh priority).

Because true "refresh urgency" isn't a directly observed column in server logs, this label comes from a defined proxy rule based on observed outcomes—specifically combining recent traffic decay (percentage drop in organic clicks over 90 days), position loss on primary keywords, and search volume potential.

In [6]:
import pandas as pd
import numpy as np

# Verify proxy target distribution behavior
np.random.seed(42)
traffic_decay = np.random.uniform(0, 0.8, 100)
search_vol = np.random.randint(100, 50000, 100)
proxy_target = traffic_decay * np.log1p(search_vol)

target_df = pd.DataFrame({
    'traffic_decay': traffic_decay,
    'search_vol': search_vol,
    'target_proxy_score': proxy_target
})

print("--- Sample Target Proxy Statistics ---")
print(target_df['target_proxy_score'].describe())


--- Sample Target Proxy Statistics ---
count    100.000000
mean       3.704567
std        2.393952
min        0.039413
25%        1.650654
50%        3.629720
75%        5.952227
max        8.122582
Name: target_proxy_score, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The primary evaluation metric is **NDCG@K (Normalized Discounted Cumulative Gain)**, specifically targeting **NDCG@20**.

Because editorial teams have limited capacity and can only review a handful of pages per week, ranking accuracy at the top of the queue matters most. An NDCG@20 score above 0.82 indicates that high-value opportunity pages are successfully positioned at the top of the list.

In [7]:
from sklearn.metrics import ndcg_score
import numpy as np

# Synthetic baseline calculation for ranking precision check
y_true = np.array([[10, 8, 7, 5, 2, 1, 0, 0]])
y_pred_model = np.array([[9.5, 8.1, 6.8, 5.2, 3.0, 1.1, 0.2, 0.1]])

score = ndcg_score(y_true, y_pred_model, k=5)
print(f"NDCG Verification Metric Check: {score:.4f}")


NDCG Verification Metric Check: 1.0000


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

* **Unit of Analysis:** **1 Row = 1 Unique URL / Page Path**.
* Each record represents a single content asset alongside its historical performance metrics, traffic trends, and structural features over time.

In [8]:
import pandas as pd

# Load starter dataset slice or build representative dataframe structure
data_path = 'flyrank/flyrank-data/starter_data.csv'

try:
    df = pd.read_csv(data_path)
except FileNotFoundError:
    # Starter structure matching unit of analysis
    df = pd.DataFrame({
        'url_path': ['/blog/ml-guide', '/blog/python-basics', '/pricing', '/docs/api'],
        'impressions_90d': [45000, 12000, 89000, 31000],
        'traffic_drop_pct': [0.35, 0.05, 0.12, 0.55],
        'days_since_last_update': [420, 120, 45, 600],
        'target_opportunity_score': [88.5, 22.1, 41.0, 94.2]
    })

print(f"Dataframe Shape (Rows, Columns): {df.shape}\n")
print("Unit of Analysis Display (1 Row = 1 Page Path):")
df.head()


Dataframe Shape (Rows, Columns): (4, 5)

Unit of Analysis Display (1 Row = 1 Page Path):


,url_path,impressions_90d,traffic_drop_pct,days_since_last_update,target_opportunity_score
0,/blog/ml-guide,45000,0.35,420,88.5
1,/blog/python-basics,12000,0.05,120,22.1
2,/pricing,89000,0.12,45,41.0
3,/docs/api,31000,0.55,600,94.2


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A simple heuristic or rule-based approach (e.g., `if traffic_drop > 30% and page_age > 180 days then refresh`) breaks down in real-world content operations due to three factors:

1. **Non-Linear Interactions & Scale:** A 15% traffic decline on a high-volume page (100k searches/mo) holds vastly higher business opportunity than a 60% drop on a low-traffic niche post. Fixed rules struggle to weigh relative business impact against decay percentages across thousands of URLs.
2. **Seasonality vs. Genuine Decay:** Rule-based logic routinely flags expected seasonal slumps (e.g., holiday drops) as content issues, creating false positives.
3. **Queue Capacity Management:** Heuristics yield unranked binary buckets ("needs update" vs. "doesn't need update") that expand uncontrolled as sites scale. Machine learning generates a calibrated continuous score to fit limited operational editorial bandwidth.

In [9]:
import pandas as pd

# Demonstrating where hardcoded rules fail compared to a scored approach
sample_pages = pd.DataFrame({
    'url': ['/blog/high-impact-guide', '/blog/niche-post'],
    'traffic_drop_pct': [0.15, 0.60],          # Page 1 dropped less %
    'monthly_search_vol': [100000, 500],       # Page 1 has massive search potential
    'days_since_update': [200, 250]
})

# Naive Fixed Rule: Flag if drop > 30% and age > 180 days
sample_pages['rule_flagged'] = (sample_pages['traffic_drop_pct'] > 0.30) & (sample_pages['days_since_update'] > 180)

# Scored Opportunity Proxy: Blending volume impact with drop rate
sample_pages['ml_opportunity_score'] = sample_pages['traffic_drop_pct'] * (sample_pages['monthly_search_vol'] / 1000)

print("--- Fixed Rule vs. Scored ML Comparison ---")
print(sample_pages[['url', 'rule_flagged', 'ml_opportunity_score']])

--- Fixed Rule vs. Scored ML Comparison ---
                       url  rule_flagged  ml_opportunity_score
0  /blog/high-impact-guide         False                  15.0
1         /blog/niche-post          True                   0.3


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.